# Validation Notebook (Scenario 3)
Runs data quality checks on raw orders. Sets return value for downstream report task.

In [ ]:
import sys
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import session as snowpark_session

try:
    session = get_active_session()
except Exception:
    session = next(iter(snowpark_session._active_sessions))

TARGET_DB = 'AGENTHIVE_LINEAGE'
TARGET_SCHEMA = 'NPO_DEMO'

user_args = [a for a in sys.argv[1:] if not a.startswith('-') and not a.startswith('/')]
if len(user_args) > 0:
    TARGET_DB = user_args[0]

print(f"Validation notebook started. TARGET_DB={TARGET_DB}")

In [ ]:
# Run data quality checks
checks = session.sql(f"""
SELECT 
  COUNT(*) AS total_rows,
  COUNT(CASE WHEN AMOUNT IS NULL THEN 1 END) AS null_amounts,
  COUNT(CASE WHEN AMOUNT < 0 THEN 1 END) AS negative_amounts,
  COUNT(CASE WHEN CATEGORY IS NULL THEN 1 END) AS null_categories,
  COUNT(DISTINCT CATEGORY) AS distinct_categories
FROM {TARGET_DB}.{TARGET_SCHEMA}.RAW_ORDERS
""").collect()[0]

total = checks['TOTAL_ROWS']
nulls = checks['NULL_AMOUNTS']
negatives = checks['NEGATIVE_AMOUNTS']
status = 'SUCCESS' if nulls == 0 and negatives == 0 else 'FAILED'

# Write validation results
session.sql(f"""
CREATE OR REPLACE TABLE {TARGET_DB}.{TARGET_SCHEMA}.VALIDATION_RESULTS AS
SELECT 
  {total} AS TOTAL_ROWS,
  {nulls} AS NULL_AMOUNTS,
  {negatives} AS NEGATIVE_AMOUNTS,
  '{status}' AS VALIDATION_STATUS,
  CURRENT_TIMESTAMP() AS VALIDATED_AT
""").collect()

OUTPUT_TABLE = f'{TARGET_DB}.{TARGET_SCHEMA}.VALIDATION_RESULTS'
print(f"Validation: {status} ({total} rows, {nulls} nulls, {negatives} negatives)")

In [ ]:
# Set return value for downstream task
return_payload = f"{OUTPUT_TABLE}|{total}|VALIDATION|{status}"
try:
    session.sql(f"SELECT SYSTEM$SET_RETURN_VALUE('{return_payload}')").collect()
    print(f"Return value set: {return_payload}")
except Exception as e:
    if 'must be called from within a task' in str(e):
        print(f"Interactive mode - skipping (payload: {return_payload})")
    else:
        raise